# 2.16 Simulated annealing & tabu search

Metaheuristics fight local traps by controlling memory: annealing remembers temperature, tabu search remembers forbidden reversals.

Huge discrete neighborhoods make exact combinatorial search impractical. Annealing accepts occasional uphill moves, while tabu search uses short-term memory to avoid immediate cycling.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from math import exp

SEED = 216
rng = np.random.default_rng(SEED)


def annealing_accept_probability(delta, temperature):
    if delta <= 0:
        return 1.0
    return exp(-delta / temperature)


def integer_neighbors(state, lower, upper):
    state = np.asarray(state, dtype=int)
    neighbors = []
    moves = []

    for index in range(len(state)):
        for step in [-1, 1]:
            candidate = state.copy()
            candidate[index] = candidate[index] + step
            if candidate[index] >= lower[index] and candidate[index] <= upper[index]:
                neighbors.append(candidate)
                move = np.zeros(len(state), dtype=int)
                move[index] = step
                moves.append(move)

    return neighbors, moves


def simulated_annealing(objective, start, lower, upper, iterations=120, initial_temperature=2.0, cooling=0.96):
    state = np.asarray(start, dtype=int).copy()
    best_state = state.copy()
    current_loss = float(objective(state))
    best_loss = current_loss
    path = [state.copy()]
    best_curve = [best_loss]
    temperature = float(initial_temperature)

    for iteration in range(iterations):
        neighbors, moves = integer_neighbors(state, lower, upper)
        choice = int(rng.integers(0, len(neighbors)))
        candidate = neighbors[choice]
        candidate_loss = float(objective(candidate))
        delta = candidate_loss - current_loss
        probability = annealing_accept_probability(delta, temperature)

        if rng.random() < probability:
            state = candidate.copy()
            current_loss = candidate_loss

        if current_loss < best_loss:
            best_loss = current_loss
            best_state = state.copy()

        path.append(state.copy())
        best_curve.append(best_loss)
        temperature = temperature * cooling

    return {
        "best_state": best_state,
        "best_loss": best_loss,
        "path": np.asarray(path),
        "best_curve": np.asarray(best_curve),
    }


def tabu_search(objective, start, lower, upper, iterations=80, tenure=3):
    state = np.asarray(start, dtype=int).copy()
    best_state = state.copy()
    current_loss = float(objective(state))
    best_loss = current_loss
    tabu_moves = []
    path = [state.copy()]
    best_curve = [best_loss]

    for iteration in range(iterations):
        neighbors, moves = integer_neighbors(state, lower, upper)
        candidate_records = []

        for candidate, move in zip(neighbors, moves):
            reversal = tuple((-move).tolist())
            is_tabu = reversal in tabu_moves
            candidate_loss = float(objective(candidate))
            aspiration = candidate_loss < best_loss
            if not is_tabu or aspiration:
                candidate_records.append((candidate_loss, candidate, tuple(move.tolist())))

        if not candidate_records:
            tabu_moves = tabu_moves[-max(1, tenure - 1):]
            continue

        candidate_records.sort(key=lambda item: item[0])
        current_loss, state, accepted_move = candidate_records[0]
        tabu_moves.append(accepted_move)
        tabu_moves = tabu_moves[-tenure:]

        if current_loss < best_loss:
            best_loss = current_loss
            best_state = state.copy()

        path.append(state.copy())
        best_curve.append(best_loss)

    return {
        "best_state": best_state,
        "best_loss": best_loss,
        "path": np.asarray(path),
        "best_curve": np.asarray(best_curve),
    }


def hybrid_annealing_tabu(objective, start, lower, upper, iterations=120, initial_temperature=2.0, cooling=0.96, tenure=3):
    annealed = simulated_annealing(objective, start, lower, upper, iterations=iterations, initial_temperature=initial_temperature, cooling=cooling)
    polished = tabu_search(objective, annealed["best_state"], lower, upper, iterations=max(20, iterations // 3), tenure=tenure)
    best_curve = np.minimum.accumulate(np.concatenate([annealed["best_curve"], polished["best_curve"]]))
    path = np.vstack([annealed["path"], polished["path"]])

    return {
        "best_state": polished["best_state"],
        "best_loss": min(annealed["best_loss"], polished["best_loss"]),
        "path": path,
        "best_curve": best_curve,
    }


def sigmoid(scores):
    return 1.0 / (1.0 + np.exp(-np.clip(scores, -30, 30)))


def load_logistic_data(max_features=6):
    try:
        from sklearn.datasets import load_breast_cancer

        data = load_breast_cancer()
        features = data.data[:, :max_features].astype(float)
        target = data.target.astype(float)
    except Exception:
        features = rng.normal(size=(180, max_features))
        true_weights = np.linspace(-1.0, 1.0, max_features)
        target = (sigmoid(features @ true_weights) > 0.5).astype(float)

    split = int(0.7 * len(features))
    train_x = features[:split]
    valid_x = features[split:]
    train_y = target[:split]
    valid_y = target[split:]
    mean = train_x.mean(axis=0)
    scale = train_x.std(axis=0) + 1e-6
    train_x = (train_x - mean) / scale
    valid_x = (valid_x - mean) / scale
    return train_x, train_y, valid_x, valid_y


LOGISTIC_DATA = load_logistic_data(max_features=6)


def logistic_subset_validation_loss(state):
    mask = np.asarray(state, dtype=int)
    selected = np.where(mask == 1)[0]
    train_x, train_y, valid_x, valid_y = LOGISTIC_DATA

    if len(selected) == 0:
        baseline = np.full_like(valid_y, train_y.mean())
        return -float(np.mean(valid_y * np.log(baseline + 1e-9) + (1.0 - valid_y) * np.log(1.0 - baseline + 1e-9)))

    x_train = train_x[:, selected]
    x_valid = valid_x[:, selected]
    weights = np.zeros(x_train.shape[1])
    bias = 0.0

    for step in range(80):
        prediction = sigmoid(x_train @ weights + bias)
        error = prediction - train_y
        gradient = x_train.T @ error / len(train_y) + 1e-3 * weights
        bias_gradient = float(np.mean(error))
        weights = weights - 0.12 * gradient
        bias = bias - 0.12 * bias_gradient

    valid_prediction = sigmoid(x_valid @ weights + bias)
    loss = -np.mean(valid_y * np.log(valid_prediction + 1e-9) + (1.0 - valid_y) * np.log(1.0 - valid_prediction + 1e-9))
    sparsity = 0.01 * len(selected)
    return float(loss + sparsity)


def build_metaheuristic_ladder():
    def d1(state):
        x, y = state
        return (x - 1) ** 2 + (y + 1) ** 2

    def d2(state):
        x, y = state
        return 0.1 * (x - 4) ** 2 + 5.0 * (y + 2) ** 2

    def d3(state):
        x, y = state
        return (x * x + y * y) * 0.08 + 1.2 * np.sin(1.7 * x) * np.cos(1.3 * y)

    def d5(state):
        weights = np.array([2, 3, 4, 5, 6, 7, 3, 4, 8, 9])
        gains = np.array([7, 8, 9, 11, 13, 14, 6, 7, 16, 17])
        mask = np.clip(state, 0, 1)
        overweight = max(0, np.dot(mask, weights) - 22)
        diversity_bonus = 0.2 * len(np.unique(np.where(mask == 1)[0] % 3))
        return -float(np.dot(mask, gains)) + 10.0 * overweight - diversity_bonus

    return [
        {"name": "D1 discrete quadratic bowl", "objective": d1, "start": np.array([-3, 3]), "lower": np.array([-5, -5]), "upper": np.array([5, 5])},
        {"name": "D2 anisotropic discrete bowl", "objective": d2, "start": np.array([-4, 4]), "lower": np.array([-6, -6]), "upper": np.array([6, 6])},
        {"name": "D3 rugged multimodal grid", "objective": d3, "start": np.array([5, -5]), "lower": np.array([-6, -6]), "upper": np.array([6, 6])},
        {"name": "D4 real logistic feature-subset validation loss", "objective": logistic_subset_validation_loss, "start": np.array([0, 0, 0, 0, 0, 0]), "lower": np.zeros(6, dtype=int), "upper": np.ones(6, dtype=int)},
        {"name": "D5 constrained high-dimensional subset", "objective": d5, "start": np.zeros(10, dtype=int), "lower": np.zeros(10, dtype=int), "upper": np.ones(10, dtype=int)},
    ]


## The concept, built once: temperature and tabu memory

Simulated annealing accepts an uphill move of size $\Delta \gt 0$ with probability $\exp(-\Delta/T)$. Tabu search stores recent moves or reversals so the search path changes even when local objective values invite cycling.

In [ ]:

prob_t1 = annealing_accept_probability(2.0, 1.0)
prob_t5 = annealing_accept_probability(2.0, 5.0)
prob_t01 = annealing_accept_probability(2.0, 0.1)

print("T=1", prob_t1)
print("T=5", prob_t5)
print("T=0.1", prob_t01)

assert abs(prob_t1 - 0.1353352832366127) < 1e-12
assert abs(prob_t5 - 0.6703200460356393) < 1e-12
assert abs(prob_t01 - 0.000000002061153622438558) < 1e-20


## Tabu reversal uses the lesson move

If the search moves $+1$ from state $0$ to state $1$, the immediate reversal is $-1$. Storing the accepted move makes that reversal tabu until the tenure expires.

In [ ]:

accepted_move = np.array([1])
forbidden_reversal = tuple((-accepted_move).tolist())
recent_tabu = [tuple(accepted_move.tolist())]
immediate_reversal = tuple(np.array([-1]).tolist())
is_reversal_forbidden = immediate_reversal == forbidden_reversal

print("stored move", recent_tabu)
print("immediate reversal", immediate_reversal)
print("reversal forbidden", is_reversal_forbidden)

assert forbidden_reversal == (-1,)
assert is_reversal_forbidden


## The dataset ladder

D1 is a discrete bowl, D2 stretches the axes, D3 is rugged, D4 is a real NumPy logistic-regression feature-subset validation loss, and D5 is a larger constrained subset with enough local traps for memory to matter.

In [ ]:

rungs = build_metaheuristic_ladder()

for index, rung in enumerate(rungs, start=1):
    start_loss = rung["objective"](rung["start"])
    print(f"D{index}: {rung['name']}")
    print("  dimension", len(rung["start"]), "start", rung["start"].tolist())
    print("  start loss", round(start_loss, 4))


## Run the same method across D1-D5

Every rung uses the same hybrid annealing-plus-tabu method. The metric is best feasible loss; lower is better.

In [ ]:

metaheuristic_results = []

for index, rung in enumerate(rungs, start=1):
    result = hybrid_annealing_tabu(
        rung["objective"],
        rung["start"],
        rung["lower"],
        rung["upper"],
        iterations=100,
        initial_temperature=3.0,
        cooling=0.96,
        tenure=3,
    )
    metaheuristic_results.append(result)
    print(f"D{index} | best feasible loss {result['best_loss']:.5f} | iterations {len(result['best_curve'])}")


## Results visualization

The left panels show state trajectories. The right panel shows best feasible loss versus iteration.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
flat_axes = axes.ravel()

for index, result in enumerate(metaheuristic_results):
    ax = flat_axes[index]
    path = result["path"]
    if path.shape[1] == 1:
        ax.plot(path[:, 0], marker="o")
        ax.set_xlabel("iteration")
        ax.set_ylabel("state")
    else:
        ax.plot(path[:, 0], path[:, 1], marker="o", linewidth=1)
        ax.set_xlabel("state 0")
        ax.set_ylabel("state 1")
    ax.set_title(f"D{index + 1} state path")

summary_ax = flat_axes[-1]
for index, result in enumerate(metaheuristic_results, start=1):
    summary_ax.plot(result["best_curve"], label=f"D{index}")

summary_ax.set_title("best feasible loss vs iteration")
summary_ax.set_xlabel("iteration")
summary_ax.set_ylabel("best loss")
summary_ax.legend()
plt.tight_layout()


## Pitfall on D5: cooling or memory set badly

Cooling too fast collapses $\exp(-\Delta/T)$ into greedy search. Cooling too slowly wanders. Excessive tabu tenure can forbid useful moves. The fix compares schedules and a moderate tenure.

In [ ]:

d5 = rungs[-1]
fast_cooling = hybrid_annealing_tabu(
    d5["objective"],
    d5["start"],
    d5["lower"],
    d5["upper"],
    iterations=100,
    initial_temperature=3.0,
    cooling=0.50,
    tenure=3,
)
slow_cooling = hybrid_annealing_tabu(
    d5["objective"],
    d5["start"],
    d5["lower"],
    d5["upper"],
    iterations=100,
    initial_temperature=3.0,
    cooling=0.995,
    tenure=3,
)
moderate = hybrid_annealing_tabu(
    d5["objective"],
    d5["start"],
    d5["lower"],
    d5["upper"],
    iterations=100,
    initial_temperature=3.0,
    cooling=0.96,
    tenure=3,
)
long_tabu = hybrid_annealing_tabu(
    d5["objective"],
    d5["start"],
    d5["lower"],
    d5["upper"],
    iterations=100,
    initial_temperature=3.0,
    cooling=0.96,
    tenure=9,
)

print("fast cooling", fast_cooling["best_loss"])
print("slow cooling", slow_cooling["best_loss"])
print("moderate cooling", moderate["best_loss"])
print("long tabu", long_tabu["best_loss"])

assert np.isfinite(moderate["best_loss"])


## Evaluate it + Practice

- Metric: track best feasible loss against a no-skill baseline: greedy local search with zero uphill acceptance and no tabu memory.
- Cheap sanity check: acceptance probabilities for delta 2 must match 0.135335, 0.670320, and 0.00000000206.
- Ablation: set tenure to 0 or cooling to 0.5 and compare D5 curves.
- Failure signals: acceptance collapses immediately, path cycles, or tabu blocks every useful neighbor.

### Practice

- Sweep the D5 cooling rate over three values and plot final best loss.

- Change tabu tenure on D3 and inspect whether the trajectory cycles less.

- Replace random neighbor selection with all-neighbor sampling and compare budget use.